In [ ]:
import pandas as pd

df_train = pd.read_csv('binary/train_labels.csv')
df_val = pd.read_csv('binary/val_labels.csv')
df_test = pd.read_csv('binary/test_labels.csv')

In [ ]:
df_train = pd.concat([df_train, df_val])
df_train

,Unnamed: 0,id,label
0,7469,VFG016628,1
1,7273,VFG031691,1
2,724,VFG000676,1
3,11873,gi|13449090|ref|NP_085306.1|,1
4,5601,VFG045743,1
...,...,...,...
2878,9501,VFG040343,1
2879,2433,sp|P9WPN3|CP132_MYCTU,0
2880,5950,VFG008206,1
2881,728,sp|Q10284|SSB1_SCHPO,0


In [ ]:
import numpy as np
import os
import torch
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef
from xgboost import XGBClassifier
from tqdm import tqdm

def load_embeddings(df, emb_dir):
    features = []
    labels = []
    for idx in tqdm(range(len(df)), desc=f"Loading embeddings from {emb_dir}"):
        id_ = df.iloc[idx]['id']
        label = df.iloc[idx]['label']
        emb_path = os.path.join(emb_dir, f'{id_}.pt')
        emb = torch.load(emb_path)
        emb_np = emb.numpy()

        if emb_np.ndim == 2:
            emb_np = emb_np.mean(axis=0)
        elif emb_np.ndim == 1:
            pass
        else:
            raise ValueError(f'Unexpected embedding dimension for id {id_}: {emb_np.shape}')

        features.append(emb_np)
        labels.append(label)

    features = np.stack(features)
    labels = np.array(labels)
    return features, labels

def evaluate_xgb(X_train, y_train, X_test, y_test):
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'AUC': roc_auc_score(y_test, y_proba),
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred)
    }
    return metrics

def ablation_study(df_train, df_test, dir1, dir2, dir3):
    # Load embeddings individually with progress bars
    X1_train, y_train = load_embeddings(df_train, dir1)
    X2_train, _ = load_embeddings(df_train, dir2)
    X3_train, _ = load_embeddings(df_train, dir3)

    X1_test, y_test = load_embeddings(df_test, dir1)
    X2_test, _ = load_embeddings(df_test, dir2)
    X3_test, _ = load_embeddings(df_test, dir3)

    results = {}

    # 1) Performance using each embedding alone
    print("Evaluating individual embeddings...")
    for i, (X_tr, X_te, name) in enumerate([(X1_train, X1_test, 'esm2'),
                                           (X2_train, X2_test, 'interproscan'),
                                           (X3_train, X3_test, 'taxonomy')]):
        print(f"Evaluating XGB with {name} embeddings")
        results[f'Only_{name}'] = evaluate_xgb(X_tr, y_train, X_te, y_test)

    # 2) Performance when leaving one embedding out from combined embeddings
    print("Evaluating embeddings combined with one left out...")
    X_all_train = np.concatenate([X1_train, X2_train, X3_train], axis=1)
    X_all_test = np.concatenate([X1_test, X2_test, X3_test], axis=1)

    leave_out_dirs = ['esm2', 'interproscan', 'taxonomy']
    for leave_out in tqdm(leave_out_dirs, desc="Ablation leave-one-out"):
        if leave_out == 'esm2':
            X_train_abl = np.concatenate([X2_train, X3_train], axis=1)
            X_test_abl = np.concatenate([X2_test, X3_test], axis=1)
        elif leave_out == 'interproscan':
            X_train_abl = np.concatenate([X1_train, X3_train], axis=1)
            X_test_abl = np.concatenate([X1_test, X3_test], axis=1)
        elif leave_out == 'taxonomy':
            X_train_abl = np.concatenate([X1_train, X2_train], axis=1)
            X_test_abl = np.concatenate([X1_test, X2_test], axis=1)
        else:
            print("Something went wrong! initiat halt!")
            break

        results[f'Without_{leave_out}'] = evaluate_xgb(X_train_abl, y_train, X_test_abl, y_test)

    # Also include performance with all combined embeddings
    results['All_combined'] = evaluate_xgb(X_all_train, y_train, X_all_test, y_test)

    return results


/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


# after reducing duplicates

In [ ]:
esm_dir = '../VirulentHunter/esm_emb'
interproscan_dir = '../VirulentHunter/Bert_emb_interproscan_nodup_semantic'
taxonomy_dir ='../VirulentHunter/Bert_emb_gtdb'

results = ablation_study(df_train, df_test, esm_dir, interproscan_dir, taxonomy_dir)

Loading embeddings from ../VirulentHunter/esm_emb: 100%|██████████| 25950/25950 [14:43<00:00, 29.37it/s] 
Loading embeddings from ../VirulentHunter/Bert_emb_interproscan_nodup_semantic: 100%|██████████| 25950/25950 [01:08<00:00, 378.84it/s]
Loading embeddings from ../VirulentHunter/Bert_emb_gtdb: 100%|██████████| 25950/25950 [01:23<00:00, 310.12it/s]
Loading embeddings from ../VirulentHunter/esm_emb: 100%|██████████| 2884/2884 [01:25<00:00, 33.66it/s]
Loading embeddings from ../VirulentHunter/Bert_emb_interproscan_nodup_semantic: 100%|██████████| 2884/2884 [00:07<00:00, 384.95it/s]
Loading embeddings from ../VirulentHunter/Bert_emb_gtdb: 100%|██████████| 2884/2884 [00:07<00:00, 390.64it/s]


Evaluating individual embeddings...
Evaluating XGB with esm2 embeddings


/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:52:15] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Evaluating XGB with interproscan embeddings


/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:53:18] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Evaluating XGB with taxonomy embeddings


/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:54:14] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Evaluating embeddings combined with one left out...


Ablation leave-one-out:   0%|          | 0/3 [00:00<?, ?it/s]/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:55:09] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
Ablation leave-one-out:  33%|███▎      | 1/3 [01:03<02:07, 63.78s/it]/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:56:13] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
Ablation leave-one-out:  67%|██████▋   | 2/3 [02:33<01:19, 79.07s/it]/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:57:45] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
Ablation leave-one-out: 100%|██████████| 3/3 [04:39<00:00, 93.29s/it] 
/home/sjchoi/anaconda3/en

In [ ]:
results

{'Only_esm2': {'AUC': 0.9351039548631216,
  'Accuracy': 0.8640776699029126,
  'F1': 0.8632240055826936,
  'Precision': 0.8686797752808989,
  'Recall': 0.8578363384188626,
  'MCC': 0.7282120758809395},
 'Only_interproscan': {'AUC': 0.9071641617340687,
  'Accuracy': 0.8321775312066574,
  'F1': 0.8333333333333334,
  'Precision': 0.8276333789329685,
  'Recall': 0.8391123439667129,
  'MCC': 0.6644189714669996},
 'Only_taxonomy': {'AUC': 0.8962735721114727,
  'Accuracy': 0.8217753120665742,
  'F1': 0.8218988218988219,
  'Precision': 0.8213296398891967,
  'Recall': 0.8224687933425797,
  'MCC': 0.643551243121986},
 'Without_esm2': {'AUC': 0.9508919554248318,
  'Accuracy': 0.8828016643550625,
  'F1': 0.8823938761308281,
  'Precision': 0.8854748603351955,
  'Recall': 0.8793342579750347,
  'MCC': 0.765621738929407},
 'Without_interproscan': {'AUC': 0.956297935330226,
  'Accuracy': 0.8921636615811374,
  'F1': 0.892573402417962,
  'Precision': 0.8891947694425327,
  'Recall': 0.8959778085991679,
  '

In [ ]:
formatted_results = {
    outer_k: {k: round(v, 3) for k, v in outer_v.items()}
    for outer_k, outer_v in results.items()
}

print(formatted_results)


{'Only_esm2': {'AUC': 0.935, 'Accuracy': 0.864, 'F1': 0.863, 'Precision': 0.869, 'Recall': 0.858, 'MCC': 0.728}, 'Only_interproscan': {'AUC': 0.907, 'Accuracy': 0.832, 'F1': 0.833, 'Precision': 0.828, 'Recall': 0.839, 'MCC': 0.664}, 'Only_taxonomy': {'AUC': 0.896, 'Accuracy': 0.822, 'F1': 0.822, 'Precision': 0.821, 'Recall': 0.822, 'MCC': 0.644}, 'Without_esm2': {'AUC': 0.951, 'Accuracy': 0.883, 'F1': 0.882, 'Precision': 0.885, 'Recall': 0.879, 'MCC': 0.766}, 'Without_interproscan': {'AUC': 0.956, 'Accuracy': 0.892, 'F1': 0.893, 'Precision': 0.889, 'Recall': 0.896, 'MCC': 0.784}, 'Without_taxonomy': {'AUC': 0.944, 'Accuracy': 0.874, 'F1': 0.873, 'Precision': 0.881, 'Recall': 0.865, 'MCC': 0.748}, 'All_combined': {'AUC': 0.96, 'Accuracy': 0.898, 'F1': 0.898, 'Precision': 0.9, 'Recall': 0.896, 'MCC': 0.796}}


In [ ]:
formatted_results['Only_esm2']

{'AUC': 0.935,
 'Accuracy': 0.864,
 'F1': 0.863,
 'Precision': 0.869,
 'Recall': 0.858,
 'MCC': 0.728}

In [ ]:
formatted_results['Only_interproscan']

{'AUC': 0.907,
 'Accuracy': 0.832,
 'F1': 0.833,
 'Precision': 0.828,
 'Recall': 0.839,
 'MCC': 0.664}

In [ ]:
formatted_results['Only_taxonomy']

{'AUC': 0.896,
 'Accuracy': 0.822,
 'F1': 0.822,
 'Precision': 0.821,
 'Recall': 0.822,
 'MCC': 0.644}

In [ ]:
formatted_results['Without_esm2']

{'AUC': 0.951,
 'Accuracy': 0.883,
 'F1': 0.882,
 'Precision': 0.885,
 'Recall': 0.879,
 'MCC': 0.766}

In [ ]:
formatted_results['Without_interproscan']

{'AUC': 0.956,
 'Accuracy': 0.892,
 'F1': 0.893,
 'Precision': 0.889,
 'Recall': 0.896,
 'MCC': 0.784}

In [ ]:
formatted_results['Without_taxonomy']

{'AUC': 0.944,
 'Accuracy': 0.874,
 'F1': 0.873,
 'Precision': 0.881,
 'Recall': 0.865,
 'MCC': 0.748}